# Looking at the images

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/main/finished/colabs/00_image_exploration.ipynb)

Before training, I compared the appearance of the CIFAR images and the synthetic test images: color distributions, contrast, quantization and resizing. I also tried marginal RGB optimal transport to see what matching the color distributions would do.

The histogram and OT cells use aggregate test statistics for exploration. I did **not** use these statistics or the resulting transforms in any Kaggle submission. This notebook does not train a classifier.


In [ ]:
%pip -q install kagglehub
from pathlib import Path
import os
import kagglehub

# Authenticate in the Colab runtime; browser login is separate.
# If prompted, enter your Kaggle API token in the interactive input only.
# Obtain it at https://www.kaggle.com/settings/api (Generate New Token).
kagglehub.login()
download_root = Path(kagglehub.competition_download('mva-recvis-2026'))
matches = list(download_root.rglob('test.npz'))
if len(matches) != 1:
    raise RuntimeError(f'Expected one test.npz under {download_root}, found {matches}')
DATA_DIR = matches[0].parent
RUN_DIR = Path('/content/recvis_runs/baseline')
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.environ['RECVIS_INPUT_DIR'] = str(DATA_DIR)
os.environ['RECVIS_OUTPUT_DIR'] = str(RUN_DIR)
print('Data:', DATA_DIR)
print('Outputs:', RUN_DIR)
print('Files:', [p.name for p in DATA_DIR.iterdir()])


In [ ]:
# Preview 24 random test images with their IDs.
import numpy as np
import matplotlib.pyplot as plt

with np.load(DATA_DIR / 'test.npz', allow_pickle=False) as archive:
    preview_images = archive['images']
    preview_ids = archive['ids'] if 'ids' in archive.files else np.arange(len(preview_images))
print('Archive shape:', preview_images.shape)
rng = np.random.default_rng(42)  # Use None instead of 42 for a new sample each run.
indices = rng.choice(len(preview_images), size=min(24, len(preview_images)), replace=False)
fig, axes = plt.subplots(4, 6, figsize=(12, 8))
for ax in axes.flat:
    ax.axis('off')
for ax, index in zip(axes.flat, indices):
    ax.imshow(preview_images[index], interpolation='nearest')
    ax.set_title(f'ID {preview_ids[index]}', fontsize=9)
plt.tight_layout()
plt.show()
del preview_images


## RGB histograms

First, the global histogram for each channel, followed by six individual images. The log scale makes small peaks easier to see. I wanted to check whether the unusual appearance could be related to quantization, although a histogram alone cannot establish how the images were generated.


In [ ]:
# Raw RGB histograms: all test pixels + six random individual images.
# Count the original uint8 pixels before preprocessing.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

with np.load(DATA_DIR / 'test.npz', allow_pickle=False) as archive:
    hist_images = archive['images']
    hist_ids = archive['ids'] if 'ids' in archive.files else np.arange(len(hist_images))
assert hist_images.dtype == np.uint8, hist_images.dtype
assert hist_images.ndim == 4 and hist_images.shape[-1] == 3, hist_images.shape
counts = np.stack([np.bincount(hist_images[..., c].ravel(), minlength=256) for c in range(3)])
assert np.all(counts.sum(axis=1) == np.prod(hist_images.shape[:3]))
levels = np.arange(256)
colors = ['red', 'green', 'blue']
channels = ['R', 'G', 'B']
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for c, (color, name) in enumerate(zip(colors, channels)):
    fraction = counts[c] / counts[c].sum()
    axes[0, c].bar(levels, fraction, width=1, color=color)
    axes[0, c].set_title(f'{name}: all test pixels')
    axes[0, c].set_ylabel('Pixel fraction')
    axes[1, c].bar(levels, np.maximum(counts[c], 0.5), width=1, color=color)
    axes[1, c].set_yscale('log')
    axes[1, c].set_ylabel('Count (log; zero shown at 0.5)')
    axes[1, c].set_xlabel('Original channel value [0, 255]')
    axes[1, c].set_xlim(-0.5, 255.5)
plt.tight_layout()
plt.show()

summary = []
for c, name in enumerate(channels):
    probabilities = counts[c] / counts[c].sum()
    nonzero = probabilities > 0
    top = np.argsort(counts[c])[-10:][::-1]
    summary.append(dict(channel=name, occupied_levels=int(nonzero.sum()),
                        entropy_bits=float(-np.sum(probabilities[nonzero] * np.log2(probabilities[nonzero]))),
                        fraction_at_0=float(probabilities[0]), fraction_at_255=float(probabilities[255]),
                        most_frequent_values=top.tolist()))
display(pd.DataFrame(summary))

selected = np.random.default_rng(42).choice(len(hist_images), min(6, len(hist_images)), replace=False)
fig, axes = plt.subplots(len(selected), 4, figsize=(15, 2.3 * len(selected)), squeeze=False)
for row, index in enumerate(selected):
    im = hist_images[index]
    axes[row, 0].imshow(im, interpolation='nearest')
    axes[row, 0].set_title(f'Test ID {hist_ids[index]}')
    axes[row, 0].axis('off')
    for c, (color, name) in enumerate(zip(colors, channels)):
        local_counts = np.bincount(im[..., c].ravel(), minlength=256)
        axes[row, c + 1].bar(levels, local_counts / local_counts.sum(), width=1, color=color)
        axes[row, c + 1].set_xlim(-0.5, 255.5)
        axes[row, c + 1].set_title(f'{name}: {np.count_nonzero(local_counts)} occupied levels')
        axes[row, c + 1].set_xlabel('Intensity')
plt.tight_layout()
plt.show()
print(f'Global histograms use all {len(hist_images):,} images; only {len(selected)} images are displayed.')
print('Regularly spaced peaks may support quantization. Histograms alone cannot identify the generating process.')
del hist_images


## Marginal RGB optimal transport

I match train and test histograms separately for each RGB channel using monotone 1D OT with squared intensity cost. The lookup table is the barycentric projection of each coupling. This ignores correlations between channels, and rounding to uint8 means the transformed histogram need not match the target exactly.

The current preview applies the train-to-test map to 16 **test** images as a visual check. To apply the forward map to its source distribution, use `DISPLAY_SPLIT='train'`. `MAP_DIRECTION='test_to_train'` gives the reverse transport; this is not generally the inverse of the forward lookup table.

I left this as an exploratory experiment and did not include OT in the training or inference pipelines. Reference: [POT user guide](https://pythonot.github.io/user_guide.html).


In [ ]:
# Global histograms of raw uint8 pixels; retain only 16 random images per split.
# Work with raw uint8 pixels.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def color_histogram_and_sample(path, sample_size=16, seed=42):
    with np.load(path, allow_pickle=False) as archive:
        images = archive['images']
        assert images.dtype == np.uint8 and images.ndim == 4 and images.shape[-1] == 3
        indices = np.random.default_rng(seed).choice(
            len(images), min(sample_size, len(images)), replace=False)
        preview = images[indices].copy()
        ids = archive['ids'][indices] if 'ids' in archive.files else indices.copy()
        histogram = np.zeros((3, 256), dtype=np.int64)
        for start in range(0, len(images), 1024):
            batch = images[start:start + 1024]
            for channel in range(3):
                histogram[channel] += np.bincount(batch[..., channel].ravel(), minlength=256)
        pixel_count = int(np.prod(images.shape[:3]))
        assert np.all(histogram.sum(axis=1) == pixel_count)
        shape = images.shape
    print(f'{Path(path).name}: {shape}, {pixel_count:,} pixels per channel')
    return histogram, preview, ids

ot_train_counts, ot_train_sample, ot_train_ids = color_histogram_and_sample(DATA_DIR / 'train.npz')
ot_test_counts, ot_test_sample, ot_test_ids = color_histogram_and_sample(DATA_DIR / 'test.npz')
ot_train_prob = ot_train_counts / ot_train_counts.sum(axis=1, keepdims=True)
ot_test_prob = ot_test_counts / ot_test_counts.sum(axis=1, keepdims=True)
ot_levels = np.arange(256, dtype=np.float64)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for channel, name in enumerate('RGB'):
    axes[channel].plot(ot_levels, ot_train_prob[channel], label='Train', alpha=0.85)
    axes[channel].plot(ot_levels, ot_test_prob[channel], label='Test', alpha=0.85)
    axes[channel].set(title=f'{name}: raw global histograms', xlabel='Intensity [0,255]',
                      ylabel='Pixel probability')
    axes[channel].legend()
plt.tight_layout()
plt.show()


In [ ]:
# Exact monotone 1D coupling, separately for each RGB marginal.
# Each channel is matched independently.
# Interval overlaps implement the common-quantile coupling, optimal for squared cost.
def monotone_plan(source, target):
    source = np.asarray(source, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    if source.ndim != 1 or target.ndim != 1:
        raise ValueError('Expected 1D histograms')
    if np.any(source < 0) or np.any(target < 0) or source.sum() <= 0 or target.sum() <= 0:
        raise ValueError('Histograms must be nonnegative with positive total mass')
    source = source / source.sum()
    target = target / target.sum()
    end_source, end_target = np.cumsum(source), np.cumsum(target)
    end_source[-1], end_target[-1] = 1.0, 1.0
    start_source, start_target = np.r_[0.0, end_source[:-1]], np.r_[0.0, end_target[:-1]]
    plan = np.maximum(0.0,
        np.minimum(end_source[:, None], end_target[None, :])
        - np.maximum(start_source[:, None], start_target[None, :]))
    np.testing.assert_allclose(plan.sum(axis=1), source, atol=1e-12, rtol=0)
    np.testing.assert_allclose(plan.sum(axis=0), target, atol=1e-12, rtol=0)
    return plan

def barycentric_lut(plan):
    # Deterministic conditional mean of the exact coupling.
    # Atom splitting means this map need not reproduce the target histogram exactly.
    masses = plan.sum(axis=1)
    supported = masses > 0
    source_values = np.arange(plan.shape[0], dtype=np.float64)
    target_values = np.arange(plan.shape[1], dtype=np.float64)
    means = (plan @ target_values)[supported] / masses[supported]
    # The plan does not define a map on absent source bins.
    # Extend monotonically by linear interpolation, with constant endpoints.
    mapping = np.interp(source_values, source_values[supported], means)
    assert np.all(np.diff(mapping) >= -1e-8)
    return np.clip(np.rint(mapping), 0, plan.shape[1] - 1).astype(np.uint8)

def mapped_histogram(probabilities, mapping):
    return np.bincount(mapping.astype(np.int64), weights=probabilities, minlength=256)

def wasserstein2_1d(source, target):
    coupling = monotone_plan(source, target)
    grid = np.arange(len(source), dtype=np.float64)
    cost = (grid[:, None] - grid[None, :]) ** 2
    return float(np.sqrt(np.sum(coupling * cost)))

ot_plans = np.stack([monotone_plan(p, q) for p, q in zip(ot_train_prob, ot_test_prob)])
ot_train_to_test = np.stack([barycentric_lut(plan) for plan in ot_plans])
# Reverse conditional barycenters, not the inverse of the rounded forward LUT.
ot_test_to_train = np.stack([barycentric_lut(plan.T) for plan in ot_plans])

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
ot_metrics = []
for channel, name in enumerate('RGB'):
    transported_train = mapped_histogram(ot_train_prob[channel], ot_train_to_test[channel])
    axes[0, channel].plot(ot_levels, ot_levels, '--', color='gray', label='Identity')
    axes[0, channel].plot(ot_levels, ot_train_to_test[channel], label='Train to test')
    axes[0, channel].plot(ot_levels, ot_test_to_train[channel], label='Test to train')
    axes[0, channel].set(title=f'{name}: barycentric intensity maps',
                         xlabel='Input intensity', ylabel='Output intensity')
    axes[0, channel].legend()
    axes[1, channel].plot(ot_levels, ot_train_prob[channel], alpha=0.5, label='Train original')
    axes[1, channel].plot(ot_levels, ot_test_prob[channel], label='Target: test')
    axes[1, channel].plot(ot_levels, transported_train, label='Mapped train (rounded LUT)', alpha=0.8)
    axes[1, channel].set(xlabel='Intensity', ylabel='Pixel probability')
    axes[1, channel].legend()
    ot_metrics.append({
        'channel': name,
        'W2_train_test_intensity_units': wasserstein2_1d(ot_train_prob[channel], ot_test_prob[channel]),
        'W2_mapped_train_test_intensity_units': wasserstein2_1d(transported_train, ot_test_prob[channel]),
        'train_occupied_bins': int(np.count_nonzero(ot_train_counts[channel])),
        'test_occupied_bins': int(np.count_nonzero(ot_test_counts[channel])),
        'test_mass_on_absent_train_bins': float(ot_test_prob[channel][ot_train_counts[channel] == 0].sum()),
    })
plt.tight_layout()
plt.show()
display(pd.DataFrame(ot_metrics))
print('Exact marginal OT coupling; deterministic barycentric + uint8 mapping is an approximation.')


In [ ]:
# Preview the train-to-test color map on 16 test images.
DISPLAY_SPLIT = 'test'  # 'train' applies the forward map on its actual source.
MAP_DIRECTION = 'train_to_test'  # Use 'test_to_train' to adapt TEST colors toward TRAIN.
assert DISPLAY_SPLIT in ('train', 'test')
assert MAP_DIRECTION in ('train_to_test', 'test_to_train')
source_split = 'train' if MAP_DIRECTION == 'train_to_test' else 'test'
if DISPLAY_SPLIT != source_split:
    print('Previewing the map on images from its target domain.')
    print('Train->test on TEST images is a color effect, not test->train distribution alignment.')
    print("For alignment of test to train, set MAP_DIRECTION = 'test_to_train'.")
images = ot_test_sample if DISPLAY_SPLIT == 'test' else ot_train_sample
image_ids = ot_test_ids if DISPLAY_SPLIT == 'test' else ot_train_ids
color_lut = ot_train_to_test if MAP_DIRECTION == 'train_to_test' else ot_test_to_train
transported = np.stack([color_lut[c][images[..., c]] for c in range(3)], axis=-1)
n = min(16, len(images))
fig, axes = plt.subplots((n + 3) // 4, 8, figsize=(16, 2.7 * ((n + 3) // 4)), squeeze=False)
for ax in axes.flat:
    ax.axis('off')
for index in range(n):
    row, pair = divmod(index, 4)
    for offset, image in enumerate((images[index], transported[index])):
        ax = axes[row, 2 * pair + offset]
        ax.imshow(image, interpolation='nearest', vmin=0, vmax=255)
        label = 'Original' if offset == 0 else MAP_DIRECTION.replace('_', ' ')
        ax.set_title(f'{DISPLAY_SPLIT} ID/index {image_ids[index]}\n{label}', fontsize=8)
fig.suptitle(f'{n} random {DISPLAY_SPLIT} images: original / RGB marginal transport', y=1.01)
plt.tight_layout()
plt.show()


## Resizing: 32×32 to 64×64

Compare nearest, bilinear, bicubic and Lanczos interpolation on the same six images. All panels have the same display size, with Matplotlib interpolation disabled. `UPSAMPLE_SPLIT='train'` repeats the comparison on training images.

Resizing changes the input representation without recovering detail. For the classifiers, I used a direct resize to the encoder's input resolution rather than an intermediate 64×64 image.


In [ ]:
# Compare x2 interpolation on the same six images.
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

UPSAMPLE_SPLIT = 'test'  # Set to 'train' to repeat on training images.
with np.load(DATA_DIR / f'{UPSAMPLE_SPLIT}.npz', allow_pickle=False) as archive:
    all_images = archive['images']
    selected = np.random.default_rng(42).choice(len(all_images), min(6, len(all_images)), replace=False)
    upsample_sample = all_images[selected].copy()
    sample_ids = archive['ids'][selected] if 'ids' in archive.files else selected.copy()
del all_images

resize_methods = {
    'Nearest': Image.Resampling.NEAREST,
    'Bilinear': Image.Resampling.BILINEAR,
    'Bicubic': Image.Resampling.BICUBIC,
    'Lanczos': Image.Resampling.LANCZOS,
}

def upscale_x2(image, method='Bicubic'):
    pil = Image.fromarray(image)
    return np.asarray(pil.resize((2 * pil.width, 2 * pil.height), resize_methods[method]))

fig, axes = plt.subplots(len(selected), 5, figsize=(12, 2.2 * len(selected)), squeeze=False)
for row, (identifier, original) in enumerate(zip(sample_ids, upsample_sample)):
    h, w = original.shape[:2]
    variants = [('Original', original)] + [(name, upscale_x2(original, name)) for name in resize_methods]
    for column, (name, array) in enumerate(variants):
        ax = axes[row, column]
        # Identical display footprint; no additional smoothing by matplotlib.
        ax.imshow(array, interpolation='nearest', vmin=0, vmax=255)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(f'{name}\n{array.shape[1]} x {array.shape[0]}')
        if column == 0:
            ax.set_ylabel(f'{UPSAMPLE_SPLIT} ID/index {identifier}')
fig.suptitle('Simple x2 upsampling — same random images, same display size', y=1.01)
plt.tight_layout()
plt.show()


## Quantization and contrast

I compare the same training images at 16, 8, 4 and 2 levels per RGB channel. The last column changes contrast only, to separate its visual effect from quantization. This was another check on possible augmentations, not a conclusion about the test generator.


In [ ]:
# Random training examples: original vs uniform RGB quantization.
# Keep the image sample fixed across quantization levels.
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance

with np.load(DATA_DIR / 'train.npz', allow_pickle=False) as archive:
    n = len(archive['images'])
    sample_indices = np.random.default_rng(42).choice(n, min(6, n), replace=False)
    quantization_sample = archive['images'][sample_indices].copy()

def quantize_rgb(image, levels):
    # Float arithmetic avoids uint8 overflow. Same fixed grid for all channels.
    x = image.astype(np.float32) / 255.0
    return np.rint(np.rint(x * (levels - 1)) * 255.0 / (levels - 1)).astype(np.uint8)

levels_to_compare = [16, 8, 4, 2]
titles = ['Original'] + [f'{q} levels/channel' for q in levels_to_compare] + ['Contrast x1.8']
fig, axes = plt.subplots(len(sample_indices), len(titles),
                         figsize=(13, 2.1 * len(sample_indices)), squeeze=False)
for row, (index, original) in enumerate(zip(sample_indices, quantization_sample)):
    variants = [original] + [quantize_rgb(original, q) for q in levels_to_compare]
    variants += [np.asarray(ImageEnhance.Contrast(Image.fromarray(original)).enhance(1.8))]
    for column, image in enumerate(variants):
        ax = axes[row, column]
        ax.imshow(image, interpolation='nearest', vmin=0, vmax=255)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(titles[column])
        if column == 0:
            ax.set_ylabel(f'Train index {index}')
fig.suptitle('Random training sample: RGB quantization vs contrast', y=1.01)
plt.tight_layout()
plt.show()
